<a href="https://colab.research.google.com/github/nicolasramirezperilla/DataWave-Project/blob/master/Crea_Base_CSF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1) Instalar librerias y conexión al servidor.

In [ ]:
# Importing libraries
from google.colab import auth
from google.colab import files
import pandas as pd
import numpy as np
from datetime import datetime
from dateutil import parser  # Import dateutil.parser for automatic date parsing

# Formatting for viewing tables
from google.colab import data_table
data_table.enable_dataframe_formatter()

# Authenticating Google Sheets
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

#2) Descargar información & definir parametros.

In [ ]:
# Define the workbook and sheets
input_workbook_name = 'MPF - CSF'
output_workbook_name = 'CSFBBDD'

# Open the input workbook and get the list of sheets to process from the 'inputs' sheet
input_workbook = gc.open(input_workbook_name)
inputs_sheet = input_workbook.worksheet('Inputs')
sheets_to_process = inputs_sheet.get_values('F3:F7')
sheets_to_process = [sheet[0] for sheet in sheets_to_process if sheet]

print("Sheets to process:", sheets_to_process)  # Debug print

# Get the last real value date from cell C13
last_real_value_date_str = inputs_sheet.cell(13, 3).value
last_real_value_date = parser.parse(last_real_value_date_str)
last_real_value_date_str2 = inputs_sheet.cell(14, 3).value
last_real_value_date2 = parser.parse(last_real_value_date_str2)

Sheets to process: ['Productividad', 'Ingresos', 'Tarifas', 'Costos', 'Gastos']


#3) Consolidación Base BBDD + Formato.

In [ ]:
# Initialize an empty DataFrame for consolidation
consolidated_df = pd.DataFrame(columns=[
    "Category", "Level1", "Level2", "Level3", "Level4", "Level5", "Level6", "Level7", "Level8", "Date", "Value"
])

# Function to check if a string can be parsed as a date
def is_valid_date(date_str):
    try:
        parser.parse(date_str)
        return True
    except (parser.ParserError, TypeError):
        return False

# Process each sheet
for sheet_name in sheets_to_process:
    print("Processing sheet:", sheet_name)
    sheet = input_workbook.worksheet(sheet_name)
    data = sheet.get_all_values()
    df = pd.DataFrame(data[3:], columns=data[2])  # Data starting from row 4, columns from row 3
    category = data[1][3]  # D2 value

    # Iterate through rows in the sheet
    for i, row in df.iterrows():
        if row[0]:  # Check if Level 1 (column A) is not empty
            # Iterate through columns starting from column K (index 26)
            for col_index in range(13, len(row)):
                if row[col_index]:
                    # Clean and convert value to float if possible
                    value_str = row[col_index].replace(',', '')
                    if '%' in value_str:
                        value = float(value_str.replace('%', '')) / 100
                    else:
                        try:
                            value = float(value_str)
                        except ValueError:
                            value = None  # Set to None if not a valid number
                    if isinstance(value, float):
                        value = round(value, 5)

                    # Parse date using dateutil.parser
                    date_str = data[2][col_index]
                    if is_valid_date(date_str):
                        date = parser.parse(date_str)  # Use dateutil.parser to parse the date

                        # Create a new row in the DataFrame
                        consolidated_row = pd.Series([
                            category,
                            row[0],  # Level 1
                            row[1],  # Level 2
                            row[2],  # Level 3
                            row[3],  # Level 4
                            row[4],  # Level 5
                            row[5],  # Level 6
                            row[6],  # Level 7
                            row[7],  # Level 8
                            date,   # Date
                            value   # Value
                        ], index=consolidated_df.columns)

                        # Append the row to the consolidated DataFrame
                        consolidated_df = pd.concat([consolidated_df, consolidated_row.to_frame().T], ignore_index=True)



# Convert 'Date' column to datetime
consolidated_df['Date'] = pd.to_datetime(consolidated_df['Date'])

# Ensure 'Value' column is numeric, replacing non-numeric values with NaN
consolidated_df['Value'] = pd.to_numeric(consolidated_df['Value'], errors='coerce')

# Sort values by Category, Level1, Level2, and Date for easier processing
consolidated_df.sort_values(by=['Category', 'Level1', 'Level2', 'Date'], inplace=True)

consolidated_df = consolidated_df.fillna(0)


consolidated_df = consolidated_df.pivot_table(index=['Category', 'Level1', 'Level2', 'Level3', 'Level4', 'Level5', 'Level7', 'Level8', 'Date'], columns='Level6', values='Value', aggfunc='sum').reset_index()
consolidated_df['Real_o_Proyeccion'] = np.where(
    consolidated_df['Date'] <= last_real_value_date,
    consolidated_df['Real'],
    consolidated_df['Proyec.']
)


# Diccionario de equivalencias
equivalencias = {
    "Margen de Intereses": {
        "Ingresos": ["BanRep", "BBVA", "Otros"]
    },
    "Comisiones Netas": {
        "Comisiones Pagadas": [
            "Admon Fondos", "Contrato colaboración", "Bancarios",
            "Custodia",  "Otros"
        ]
    },
    "ROFs": {
        "": ["Diferencia de Cambio"]
    },
    "Resto Ingresos Netos Ordinarios": {
        "Dividendos": ["Dividendos"],
        "Rto. Otros Productos y Cargas": ["Negocio conjunto","Recuperaciones","Riesgo Operacional","Multas","Indemnizaciones"]
    },
    "Gastos de Personal": {
        "Personal": [
            "Percep. Fijas", "Percep. Variables", "Seg.Social", "Pensiones",
            "Indemin.", "Formacion", "Beneficios", "Otros","Ajustes"
        ]
    },
    "Gastos Generales": {
        "Generales": [
            "Asistencia", "Asistencia", "Arrendamientos", "Utiles", "Tecnologia",
            "Transporte", "Publicidad", "Informes Tecnicos", "Informes Tecnicos",
            "Tercero", "Seguros", "Representacion", "Servicios", "Asociaciones",
            "Asociaciones", "Tercero", "Otros","Ajuste"
        ]
    },
    "Tributos": {
        "Tributos": ["ICA", "GMF", "Consumos", "Otros"]
    },
    "Amortizaciones": {
        "Amort": ["Equipos", "Software", "Oficinas"]
    },
    "Saneamiento Crediticio": {
        "Otros": ["Saneamiento Crediticio"]
    },
    "Pérdida Deterioro Resto de Activos": {
        "Otros": ["Pérdida Deterioro Resto de Activos"]
    },
    "Dotaciones a Provisiones": {
        "Otros": ["Dotaciones a Provisiones"]
    },
    "Resto de Resultados No Ordinarios": {
        "Otros": ["Resto de Resultados No Ordinarios"]
    },
    "Impuesto Sociedades": {
        "Otros": ["Impuesto Sociedades"]
    },
    "Intereses Minoritarios": {
        "Otros": ["Intereses Minoritarios"]
    }
}


# Invertimos el diccionario para incluir Level 2 en las claves
equivalencias_invertidas = {}
for level_9, sub_dict in equivalencias.items():
    for level_2, level_3_list in sub_dict.items():
        for level_3 in level_3_list:
            equivalencias_invertidas[(level_2, level_3)] = level_9

# Excepción para ROFs (se ignora Level 2)
rofs_dict = {v: "ROFs" for v in equivalencias["ROFs"][""]}

# Agregamos la columna 'Level 9'
def asignar_level_9(row):
    key = (row["Level2"], row["Level3"])
    if row["Level2"] == "" and row["Level3"] in rofs_dict:
        return rofs_dict[row["Level3"]]
    return equivalencias_invertidas.get(key)
consolidated_df.insert(consolidated_df.columns.get_loc("Level8") + 1, "Level9", consolidated_df.apply(asignar_level_9, axis=1))
consolidated_df['Level9'] = consolidated_df['Level9'].replace(0, "")

Processing sheet: Productividad


Se truncaron las últimas líneas 5000 del resultado de transmisión.
<ipython-input-3-c929121898bc>:52: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  row[4],  # Level 5
<ipython-input-3-c929121898bc>:53: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  row[5],  # Level 6
<ipython-input-3-c929121898bc>:54: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  row[6],  # Level 7
<ipython-input-3-c929121898bc>:55: FutureWarning: Series.__getitem__ 

Processing sheet: Ingresos


Se truncaron las últimas líneas 5000 del resultado de transmisión.
<ipython-input-3-c929121898bc>:54: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  row[6],  # Level 7
<ipython-input-3-c929121898bc>:55: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  row[7],  # Level 8
<ipython-input-3-c929121898bc>:27: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if row[col_index]:
<ipython-input-3-c929121898bc>:29: FutureWarning: Series.__getitem__ 

Processing sheet: Tarifas


Se truncaron las últimas líneas 5000 del resultado de transmisión.
<ipython-input-3-c929121898bc>:54: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  row[6],  # Level 7
<ipython-input-3-c929121898bc>:55: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  row[7],  # Level 8
<ipython-input-3-c929121898bc>:27: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if row[col_index]:
<ipython-input-3-c929121898bc>:29: FutureWarning: Series.__getitem__ 

Processing sheet: Costos


Se truncaron las últimas líneas 5000 del resultado de transmisión.
<ipython-input-3-c929121898bc>:48: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  row[0],  # Level 1
<ipython-input-3-c929121898bc>:49: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  row[1],  # Level 2
<ipython-input-3-c929121898bc>:50: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  row[2],  # Level 3
<ipython-input-3-c929121898bc>:51: FutureWarning: Series.__getitem__ 

Processing sheet: Gastos


Se truncaron las últimas líneas 5000 del resultado de transmisión.
<ipython-input-3-c929121898bc>:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  value_str = row[col_index].replace(',', '')
<ipython-input-3-c929121898bc>:48: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  row[0],  # Level 1
<ipython-input-3-c929121898bc>:49: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  row[1],  # Level 2
<ipython-input-3-c929121898bc>:50: FutureWar

In [ ]:
consolidated_df.head()

Level6,Category,Level1,Level2,Level3,Level4,Level5,Level7,Level8,Level9,Date,Ppto,Proyec.,Rapel,RapelM,Real,Real_o_Proyeccion
0,Costos,Cobranzas,,Cobranzas,,,,VERDADERO,None,2024-01-31,0.0,NaN,NaN,NaN,-40.0,-40.0
1,Costos,Cobranzas,,Cobranzas,,,,VERDADERO,None,2024-02-29,0.0,NaN,NaN,NaN,-48.0,-48.0
2,Costos,Cobranzas,,Cobranzas,,,,VERDADERO,None,2024-03-31,0.0,NaN,NaN,NaN,-43.0,-43.0
3,Costos,Cobranzas,,Cobranzas,,,,VERDADERO,None,2024-04-30,0.0,NaN,NaN,NaN,-43.0,-43.0
4,Costos,Cobranzas,,Cobranzas,,,,VERDADERO,None,2024-05-31,0.0,NaN,NaN,NaN,-29.0,-29.0


#4) Calculos valores adicionales.

* Real_Acum_Año
* Real_o_Proyec_Acum_Año
* Ppto_Acum_Año
* Real_Ant
* Real_o_Proyec_Ant

In [ ]:
# Add columns for additional calculations
# Calculate accumulated values for the year, previous month's value, and accumulated value for the same period last year
consolidated_df['Real_Acum_Año'] = consolidated_df.groupby(['Category', 'Level1', 'Level2', 'Level3', 'Level4', 'Level5', 'Level7', 'Level8', consolidated_df['Date'].dt.year])['Real'].cumsum()
consolidated_df['Real_o_Proyec_Acum_Año'] = consolidated_df.groupby(['Category', 'Level1', 'Level2', 'Level3', 'Level4', 'Level5', 'Level7', 'Level8', consolidated_df['Date'].dt.year])['Real_o_Proyeccion'].cumsum()
consolidated_df['Ppto_Acum_Año'] = consolidated_df.groupby(['Category', 'Level1', 'Level2', 'Level3', 'Level4', 'Level5', 'Level7', 'Level8', consolidated_df['Date'].dt.year])['Ppto'].cumsum()

#Meses anteriores
consolidated_df['Real_Ant'] = consolidated_df.groupby(['Category', 'Level1', 'Level2', 'Level3', 'Level4', 'Level5', 'Level7', 'Level8'])['Real'].shift(1)
consolidated_df['Real_o_Proyec_Ant'] = consolidated_df.groupby(['Category', 'Level1', 'Level2', 'Level3', 'Level4', 'Level5', 'Level7', 'Level8'])['Real_o_Proyeccion'].shift(1)


#5) Merge valores Last_Year.

In [ ]:
# Create a shifted DataFrame for the previous year accumulated values
consolidated_df['Year'] = consolidated_df['Date'].dt.year
consolidated_df['Month'] = consolidated_df['Date'].dt.month


last_year_df = consolidated_df.copy()
last_year_df['Year'] += 1

# Merge to get last year's accumulated values
consolidated_df = consolidated_df.merge(
    last_year_df[['Category', 'Level1', 'Level2', 'Level3', 'Level4', 'Level5',  'Level7', 'Level8', 'Month', 'Year', 'Real_Acum_Año','Real_o_Proyec_Acum_Año','Real_o_Proyeccion']],
    left_on=['Category', 'Level1', 'Level2', 'Level3', 'Level4', 'Level5',  'Level7', 'Level8', 'Month', 'Year'],
    right_on=['Category', 'Level1', 'Level2', 'Level3', 'Level4', 'Level5',  'Level7', 'Level8', 'Month', 'Year'],
    suffixes=('', '_Last_Year'),
    how='left'
)

consolidated_df['Real_o_Proyeccion_12M'] = consolidated_df['Real_o_Proyeccion_Last_Year']
consolidated_df['Real_Acum_Last_Year'] = consolidated_df['Real_Acum_Año_Last_Year']
consolidated_df['Real_o_Proyec_Acum_Last_Year'] = consolidated_df['Real_o_Proyec_Acum_Año_Last_Year']
consolidated_df.drop(columns=['Real_Acum_Año_Last_Year','Real_o_Proyec_Acum_Año_Last_Year' ,'Real_o_Proyeccion_Last_Year','Year', 'Month'], inplace=True)

#6) Calculo metricas.

* MoM_Abs
* MoM_Porc.
* YoY_Porc.
* Cump_Ppto_Abs
* Cump_Ppto_Porc
* Cump_Ppto_Abs_Acum
* Cump_Ppto_Porc_Acum

In [ ]:
#Diferencias
consolidated_df['MoM_Abs'] = round(consolidated_df['Real'] - consolidated_df['Real_Ant'],2)
consolidated_df['MoM_Porc.'] = round((consolidated_df['MoM_Abs'] / consolidated_df['Real_Ant']) * 100,2)

consolidated_df['YoY_Abs'] = round(consolidated_df['Real_Acum_Año'] - consolidated_df['Real_Acum_Last_Year'],2)
consolidated_df['YoY_Porc.'] = round((consolidated_df['YoY_Abs'] / consolidated_df['Real_Acum_Last_Year']) * 100,2)

consolidated_df['Cump_Proy_Abs'] = round(consolidated_df['Real'] - consolidated_df['Proyec.'],2)
consolidated_df['Cump_Proy_Porc.'] = round((consolidated_df['Cump_Proy_Abs'] / consolidated_df['Proyec.']) * 100,2)

consolidated_df['Cump_Ppto_Abs'] = round(consolidated_df['Real'] - consolidated_df['Ppto'],2)
consolidated_df['Cump_Ppto_Porc.'] = np.where(
    consolidated_df['Ppto'] < 0,
    round((1 - (consolidated_df['Real'] / consolidated_df['Ppto'] - 1))*100, 2),
    round((consolidated_df['Real'] / consolidated_df['Ppto']) * 100, 2)
)

consolidated_df['Cump_Ppto_Abs_Acum'] = round(consolidated_df['Real_Acum_Año'] - consolidated_df['Ppto_Acum_Año'],2)
consolidated_df['Cump_Ppto_Porc_Acum'] = np.where(
    consolidated_df['Ppto'] < 0,
    round((1 - (consolidated_df['Real_Acum_Año'] / consolidated_df['Ppto_Acum_Año'] - 1))*100, 2),
    round((consolidated_df['Real_Acum_Año'] / consolidated_df['Ppto_Acum_Año']) * 100, 2)
)

#7) Formato JSON + Actualizar hojas de cálculo en Google Sheets.

In [ ]:
consolidated_df=consolidated_df.fillna("")

# Replace non-JSON-compliant float values with None
consolidated_df.replace([float('inf'), float('-inf'), float('nan')], None, inplace=True)

# Convert 'Date' column to string to avoid JSON serialization issues
consolidated_df['Date'] = consolidated_df['Date'].dt.strftime('%Y-%m-%d')

#Crea marca de ultima fecha real
consolidated_df['Marca_Fecha'] = ((consolidated_df['Date'] == last_real_value_date.strftime('%Y-%m-%d'))).astype(int)
consolidated_df['Marca_Fecha2'] = ((consolidated_df['Date'] == last_real_value_date2.strftime('%Y-%m-%d'))).astype(int)

# Create a new workbook and write the consolidated data
output_sheet = gc.open(output_workbook_name).worksheet('BBDD')
output_sheet.clear()
output_sheet.update([consolidated_df.columns.values.tolist()] + consolidated_df.fillna('').values.tolist())

print('Proceso_Terminado')

Proceso_Terminado
